# 00 - WRDS Setup

This notebook sets up the WRDS (Wharton Research Data Services) connection for the entire credit-risk project.

**All major data (ratings, financial statements, bankruptcy events, stock data) will be queried directly from WRDS.**

Credentials are read from `~/.pgpass` (never stored in code).

Run this notebook first. It uses a direct (non-interactive) connection so it works reliably inside Jupyter.

In [1]:
import wrds
import pandas as pd
from pathlib import Path
import sys

# Make src/ importable in later notebooks
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/hananau/Documents/Projects/capiq-credit-risk


In [2]:
# === WRDS Connection ===
# Connects via ~/.pgpass (no password in code).
# If this is your first time, run:  python src/wrds_utils.py
# to verify your .pgpass setup.

try:
    conn = wrds.Connection(autoconnect=True)
    print("✅ Successfully connected to WRDS")
    print(f"   Username : {conn._username}")
    print(f"   Host     : {conn._hostname}:{conn._port}")
except Exception as e:
    print("❌ Connection failed:")
    print(e)
    print("\nMake sure ~/.pgpass is set up correctly.")
    print("See src/wrds_utils.py for setup instructions.")
    raise


Loading library list...
Done
✅ Successfully connected to WRDS


In [3]:
# List libraries you have access to (important for credit risk work)
libs = conn.list_libraries()
print(f"You have access to {len(libs)} libraries.")
print("Relevant libraries for credit risk:")
for lib in sorted(libs):
    if any(x in lib for x in ["comp", "crsp", "wrds", "bankrupt", "rating", "fama"]):
        print("   -", lib)

You have access to 329 libraries.
Relevant libraries for credit risk:
   - ciqsamp_ratings
   - comp
   - comp_bank_daily
   - comp_execucomp
   - comp_filings
   - comp_global_daily
   - comp_na_daily_all
   - comp_segments_hist_daily
   - compsamp
   - compsamp_all
   - compsamp_computext
   - compsamp_snapshot
   - compseg
   - crsp
   - crsp_a_ccm
   - crsp_a_indexes
   - crsp_a_stock
   - crsp_a_treasuries
   - crsp_q_mutualfunds
   - crspsamp
   - crspsamp_all
   - crspsamp_mf
   - execcomp
   - pitchbk_companies_deals
   - wrds_abs_samp
   - wrds_environmental_samp
   - wrds_insiders_samp
   - wrds_mutualfund_samp
   - wrds_sec_search
   - wrds_shortvolume_samp
   - wrdsapps
   - wrdsapps_backtest_basic
   - wrdsapps_backtest_plus
   - wrdsapps_bondret
   - wrdsapps_eushort
   - wrdsapps_evtstudy_int
   - wrdsapps_evtstudy_int_ginsight
   - wrdsapps_evtstudy_lr
   - wrdsapps_evtstudy_lr_ciq
   - wrdsapps_evtstudy_us
   - wrdsapps_finratio
   - wrdsapps_finratio_ccm
   - wrdsapps

## Key WRDS Libraries We Will Use

- `comp` — Compustat (funda, fundq, adsprate for S&P ratings, etc.)
- `crsp` — CRSP (stock prices, delistings as bankruptcy proxy)
- `wrdscompa` — WRDS CCM link table (Compustat-CRSP merged)
- Bankruptcy / ratings tables as available under your subscription

All future notebooks (`01_*.ipynb` onward) will import and use the same connection pattern.

In [4]:
# Save credentials in ~/.pgpass so you never need to type the password again
print("Creating .pgpass file for passwordless future connections...")
try:
    conn.create_pgpass_file()
    print("✅ .pgpass created. Future connections can be made with just wrds_username.")
except Exception as e:
    print("Note: pgpass creation skipped or failed (you can run conn.create_pgpass_file() manually later).")
    print(e)

Creating .pgpass file for passwordless future connections...
✅ .pgpass created. Future connections can be made with just wrds_username.


## Next

Proceed to the next notebook to start extracting ratings + bankruptcy events from WRDS.

You can reuse the connection in any notebook with the same two lines.